# MULDE Hiera-L Feature Extraction (ShanghaiTech or Avenue)

This notebook extracts 1152-D spatiotemporal features from a video-anomaly dataset using **Hiera-L (Large)** for downstream **MULDE** training. It supports both supported datasets via a single selector at the top.

### Pipeline Overview
1. **Dataset selector** — choose `ShanghaiTech` or `Avenue`.
2. **Google Drive integration** — mount Drive and resolve dataset-specific paths.
3. **Environment & pinned dependencies** — `timm`, Hiera (pinned commit), `decord`, OpenCV, pandas, tqdm.
4. **Dataset scanning** — discover raw videos/frames and validate structure (handles ShanghaiTech's JPG-frame test split and Avenue's `.avi`-only layout, plus optional Avenue ground-truth labels).
5. **Loading, preprocessing & saving helpers** — natural sort, unified video decoder (decord + OpenCV fallback), JPG-frame loader, centered 16-frame / stride-4 clip sampler, Hiera normalization, atomic NPZ saver.
6. **Model setup** — Hiera-L from PyTorch Hub with `head = Identity`.
7. **Smoke test** — end-to-end validation on a single item per split.
8. **Resumable full extraction** — skip existing valid `.npz` files, build train/test manifests.
9. **Streaming train statistics** — global mean/std for standardization.
10. **MULDE-ready config index** — consolidated `extraction_config.json`.

*References*:
- **MULDE Paper**: [Micorek et al., CVPR 2024](https://ar5iv.labs.arxiv.org/html/2403.14497v1)
- **Hiera Repository**: [facebookresearch/hiera](https://github.com/facebookresearch/hiera)

## Step 0: Select Dataset
Set `DATASET_CHOICE` to the dataset you want to process. All subsequent cells adapt automatically.

| Dataset | Train input | Test input | Archive | Default feature subdir |
|---------|-------------|------------|---------|------------------------|
| `ShanghaiTech` | `.avi` videos | JPG frame folders | `shanghaitech.tar.gz` | `features/hiera_large_16x224_mae_k400_ft_k400_centered_s4` |
| `Avenue` | `.avi` videos | `.avi` videos | `Avenue_Dataset.zip` | `features/avenue_features` |

The feature subdirectory name is intentionally kept **dataset-specific** so the two datasets do not collide — `MULDE_Training_GMM.ipynb` already handles the differing naming schemes.

In [ ]:
import os
import random
import numpy as np
import torch

# ======================== SELECT DATASET ========================
DATASET_CHOICE = "ShanghaiTech"   # "ShanghaiTech" or "Avenue"
assert DATASET_CHOICE in {"ShanghaiTech", "Avenue"}, \
    f"DATASET_CHOICE must be 'ShanghaiTech' or 'Avenue', got {DATASET_CHOICE!r}"

# ======================== DRIVE / WORKSPACE ========================
DRIVE_ROOT = "/content/drive/MyDrive/MULDE"

try:
    from google.colab import drive
    print("Colab environment detected. Mounting Google Drive...")
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Google Colab. Skipping Google Drive mount.")

# ======================== DATASET-SPECIFIC RAW INPUTS ========================
if DATASET_CHOICE == "ShanghaiTech":
    # ShanghaiTech ships as a tarball; train = .avi, test = JPG frame folders.
    ARCHIVE_PATH = "/content/drive/MyDrive/shanghaitech.tar.gz"
    ARCHIVE_TYPE = "tar"   # "tar" or "zip"
    EXTRACT_ROOT = "/content/shanghaitech"
    TRAIN_VIDEO_DIR = os.path.join(EXTRACT_ROOT, "training/videos")
    TEST_FRAME_DIR  = os.path.join(EXTRACT_ROOT, "testing/frames")
    TEST_VIDEO_DIR   = None  # ShanghaiTech test is JPG frames, not videos
    GROUND_TRUTH_DIR = None  # ShanghaiTech GT lives inside the tarball, not matched here
    # Feature output subdirectory (kept dataset-specific per project convention)
    FEATURE_SUBDIR = "features/hiera_large_16x224_mae_k400_ft_k400_centered_s4"
else:
    # Avenue ships as a zip; train and test are both .avi; GT labels optional.
    ARCHIVE_PATH = "/content/drive/MyDrive/Avenue_Dataset.zip"
    ARCHIVE_TYPE = "zip"
    EXTRACT_ROOT = "/content/avenue_dataset"
    TRAIN_VIDEO_DIR = "/content/Avenue Dataset/training_videos"   # auto-discovered if missing
    TEST_VIDEO_DIR  = "/content/Avenue Dataset/testing_videos"    # auto-discovered if missing
    TEST_FRAME_DIR  = None  # Avenue test is videos, not JPG frames
    GROUND_TRUTH_DIR = "/content/drive/MyDrive/ground_truth_avenue"  # optional .npy labels
    FEATURE_SUBDIR = "features/avenue_features"

FORCE_REEXTRACT = False

# ======================== EXTRACTED FEATURE OUTPUTS ========================
FEATURE_DIR        = os.path.join(DRIVE_ROOT, FEATURE_SUBDIR)
TRAIN_FEATURE_DIR  = os.path.join(FEATURE_DIR, "train")
TEST_FEATURE_DIR   = os.path.join(FEATURE_DIR, "test")

MANIFEST_TRAIN     = os.path.join(FEATURE_DIR, "manifest_train.csv")
MANIFEST_TEST      = os.path.join(FEATURE_DIR, "manifest_test.csv")
FEATURE_STATS_PATH = os.path.join(FEATURE_DIR, "train_feature_stats.npz")
CONFIG_PATH        = os.path.join(FEATURE_DIR, "extraction_config.json")

os.makedirs(TRAIN_FEATURE_DIR, exist_ok=True)
os.makedirs(TEST_FEATURE_DIR, exist_ok=True)

# ======================== MODEL / FEATURE CONFIG ========================
HIERA_COMMIT       = "b12b842542ee5c757fcfec8c41f6b56fcbe89b65"
MODEL_NAME         = "hiera_large_16x224"
CHECKPOINT_NAME    = "mae_k400_ft_k400"
NUM_FRAMES_PER_CLIP = 16
TEMPORAL_STRIDE    = 4
IMAGE_SIZE         = 224

# ======================== REPRODUCIBILITY ========================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    print(f"Random seed set to: {seed}")

seed_everything(42)

print(f"\n--- Workspace Paths Configuration ({DATASET_CHOICE}) ---")
print(f"Archive ({ARCHIVE_TYPE}): {ARCHIVE_PATH}")
print(f"Extract root: {EXTRACT_ROOT}")
print(f"Train videos: {TRAIN_VIDEO_DIR}")
print(f"Test videos:  {TEST_VIDEO_DIR}")
print(f"Test frames:  {TEST_FRAME_DIR}")
print(f"Ground truth: {GROUND_TRUTH_DIR}")
print(f"Feature storage: {FEATURE_DIR}")

## Step 1: Extract Dataset Archive
Extracts the dataset archive into the Colab local filesystem. Skips extraction if the target directory already has content (unless `FORCE_REEXTRACT = True`). For Avenue, this cell also auto-discovers the train/test video directories if the defaults do not match your zip's layout, and locates the optional ground-truth label folder.

In [ ]:
import os
import glob
import shutil
import zipfile
import tarfile


def directory_has_content(path):
    if not os.path.isdir(path):
        return False
    with os.scandir(path) as entries:
        return any(entries)


def find_split_dir_with_avi(root, keywords):
    """Auto-discover the most likely split directory containing .avi files.
    Used by the Avenue path when the zip layout differs from the default."""
    if not os.path.isdir(root):
        return None
    candidates = []
    for dirpath, _, filenames in os.walk(root):
        avi_count = sum(1 for name in filenames if name.lower().endswith(".avi"))
        if avi_count == 0:
            continue
        path_text = dirpath.replace("\\", "/").lower()
        base_text = os.path.basename(dirpath).lower()
        if any(k in path_text or k in base_text for k in keywords):
            candidates.append((avi_count, len(dirpath), dirpath))
    if not candidates:
        return None
    candidates.sort(key=lambda item: (-item[0], item[1]))
    return candidates[0][2]


def find_ground_truth_dir(root, preferred_dir):
    candidates = [
        preferred_dir,
        os.path.join("/content/drive/MyDrive", "ground_truth_avenue"),
        os.path.join(DRIVE_ROOT, "ground_truth_avenue"),
        os.path.join(root, "ground_truth_avenue"),
    ]
    for candidate in candidates:
        if candidate and os.path.isdir(candidate) and glob.glob(os.path.join(candidate, "*.npy")):
            return candidate
    if os.path.isdir(root):
        for dirpath, _, filenames in os.walk(root):
            has_npy = any(name.lower().endswith(".npy") for name in filenames)
            path_text = dirpath.replace("\\", "/").lower()
            looks_like_gt = any(token in path_text for token in ["ground", "truth", "label", "gt"])
            if has_npy and looks_like_gt:
                return dirpath
    return preferred_dir


if FORCE_REEXTRACT and os.path.isdir(EXTRACT_ROOT):
    print(f"FORCE_REEXTRACT=True. Removing existing extraction root: {EXTRACT_ROOT}")
    shutil.rmtree(EXTRACT_ROOT)

if directory_has_content(EXTRACT_ROOT):
    print(f"{DATASET_CHOICE} archive already extracted at: {EXTRACT_ROOT}")
else:
    if not os.path.exists(ARCHIVE_PATH):
        raise FileNotFoundError(
            f"{DATASET_CHOICE} archive not found at {ARCHIVE_PATH}. "
            f"Upload it to MyDrive or edit ARCHIVE_PATH."
        )
    os.makedirs(EXTRACT_ROOT, exist_ok=True)
    print(f"Extracting {ARCHIVE_PATH} to {EXTRACT_ROOT}...")
    if ARCHIVE_TYPE == "tar":
        with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
            archive.extractall(EXTRACT_ROOT)
    else:  # zip
        with zipfile.ZipFile(ARCHIVE_PATH, "r") as archive:
            archive.extractall(EXTRACT_ROOT)
    print("Extraction complete.")

# Auto-discover split directories for Avenue if defaults are missing.
if DATASET_CHOICE == "Avenue":
    if TRAIN_VIDEO_DIR is None or not os.path.isdir(TRAIN_VIDEO_DIR):
        TRAIN_VIDEO_DIR = find_split_dir_with_avi(EXTRACT_ROOT, ["train", "training"])
    if TEST_VIDEO_DIR is None or not os.path.isdir(TEST_VIDEO_DIR):
        TEST_VIDEO_DIR = find_split_dir_with_avi(EXTRACT_ROOT, ["test", "testing"])
    GROUND_TRUTH_DIR = find_ground_truth_dir(EXTRACT_ROOT, GROUND_TRUTH_DIR)

print(f"\n--- Resolved {DATASET_CHOICE} Paths ---")
print(f"Train directory: {TRAIN_VIDEO_DIR}")
print(f"Test video dir:  {TEST_VIDEO_DIR}")
print(f"Test frame dir:  {TEST_FRAME_DIR}")
print(f"Ground truth:    {GROUND_TRUTH_DIR}")

## Step 2: Install Pinned Dependencies
Install a PyTorch-compatible `timm`, the official Hiera implementation pinned to the exact recommended commit, `decord` for fast hardware-accelerated video decoding, `opencv-python-headless`, `pandas`, and `tqdm`.

In [ ]:
# Install pinned dependencies
# Using -q flag to keep the output clean and professional
print("Installing pip dependencies. This might take a minute...")
!pip install -q timm decord opencv-python-headless pandas tqdm
!pip install -q git+https://github.com/facebookresearch/hiera.git@b12b842542ee5c757fcfec8c41f6b56fcbe89b65

print("\nAll dependencies installed successfully:")
import timm
import hiera
import decord
import cv2
import pandas as pd
import tqdm
print(f" - timm: {timm.__version__}")
print(f" - hiera: installed from commit {HIERA_COMMIT[:8]}")
print(f" - decord: {decord.__version__}")
print(f" - cv2 (OpenCV): {cv2.__version__}")
print(f" - pandas: {pd.__version__}")
print(f" - tqdm: {tqdm.__version__}")

## Step 3: Scan Dataset & Validate Structure
Discover raw inputs and validate their types. Behavior is dataset-aware:

- **ShanghaiTech**: train split is `.avi` videos; test split is folders of `.jpg` frames.
- **Avenue**: both splits are `.avi` videos; optional `.npy` ground-truth labels are matched to test videos by name.

In [ ]:
import glob
import os
import re


def natural_keys(text):
    return [int(c) if c.isdigit() else c.lower() for c in re.split(r'(\d+)', text)]


def normalize_video_key(value):
    stem = os.path.splitext(os.path.basename(str(value)))[0].lower()
    compact = re.sub(r'[^a-z0-9]+', '', stem)
    digits = re.findall(r'\d+', stem)
    if digits:
        numeric_tail = str(int(digits[-1]))
        return compact, numeric_tail
    return compact, None


def build_label_map(label_files):
    label_map = {}
    for label_path in label_files:
        stem = os.path.splitext(os.path.basename(label_path))[0]
        compact, numeric_tail = normalize_video_key(stem)
        for key in [stem.lower(), compact, numeric_tail]:
            if key:
                label_map.setdefault(key, label_path)
    return label_map


def get_label_path_for_video(video_id):
    if not label_by_video_id:
        return ""
    compact, numeric_tail = normalize_video_key(video_id)
    candidate_keys = [str(video_id).lower(), compact, numeric_tail]
    for key in candidate_keys:
        if key and key in label_by_video_id:
            return label_by_video_id[key]
    for key, label_path in label_by_video_id.items():
        if compact and key and (compact.endswith(key) or key.endswith(compact)):
            return label_path
    return ""


def scan_dataset():
    print(f"Scanning {DATASET_CHOICE} dataset directories...")

    # ---- Train split: always .avi videos ----
    if not TRAIN_VIDEO_DIR or not os.path.exists(TRAIN_VIDEO_DIR):
        print(f"Warning: Training directory not found: {TRAIN_VIDEO_DIR}")
        train_sources = []
    else:
        train_sources = sorted(glob.glob(os.path.join(TRAIN_VIDEO_DIR, "*.avi")), key=natural_keys)

    # ---- Test split: videos (Avenue) or JPG folders (ShanghaiTech) ----
    test_sources = []
    if DATASET_CHOICE == "Avenue":
        if not TEST_VIDEO_DIR or not os.path.exists(TEST_VIDEO_DIR):
            print(f"Warning: Testing video directory not found: {TEST_VIDEO_DIR}")
        else:
            test_sources = sorted(glob.glob(os.path.join(TEST_VIDEO_DIR, "*.avi")), key=natural_keys)
    else:  # ShanghaiTech: test is a directory of JPG-frame folders
        if not TEST_FRAME_DIR or not os.path.exists(TEST_FRAME_DIR):
            print(f"Warning: Testing frame directory not found: {TEST_FRAME_DIR}")
        else:
            test_sources = sorted(
                [os.path.join(TEST_FRAME_DIR, d)
                 for d in os.listdir(TEST_FRAME_DIR)
                 if os.path.isdir(os.path.join(TEST_FRAME_DIR, d))],
                key=natural_keys,
            )

    print(f"Discovered training videos (.avi): {len(train_sources)}")
    print(f"Discovered testing sources: {len(test_sources)} "
          f"({'avi' if DATASET_CHOICE == 'Avenue' else 'jpg folders'})")

    # ---- Validation assertions ----
    if len(train_sources) > 0:
        for path in train_sources:
            assert path.lower().endswith(".avi"), \
                f"Validation Error: Train path {path} must be a .avi file."
        print("✓ All training video paths end with .avi")

    if DATASET_CHOICE == "ShanghaiTech" and len(test_sources) > 0:
        for folder in test_sources:
            jpgs = glob.glob(os.path.join(folder, "*.jpg"))
            assert len(jpgs) > 0, \
                f"Validation Error: Test folder {folder} contains no JPG frames."
        print("✓ All testing folders contain at least one JPG frame sequence")

    if DATASET_CHOICE == "Avenue" and len(test_sources) > 0:
        for path in test_sources:
            assert path.lower().endswith(".avi"), \
                f"Validation Error: Test path {path} must be a .avi file."
        print("✓ All testing video paths end with .avi")

    # ---- Optional Avenue ground-truth label matching ----
    global label_by_video_id
    label_by_video_id = {}
    if DATASET_CHOICE == "Avenue":
        if GROUND_TRUTH_DIR and os.path.exists(GROUND_TRUTH_DIR):
            label_files = sorted(glob.glob(os.path.join(GROUND_TRUTH_DIR, "*.npy")), key=natural_keys)
        else:
            label_files = []
            print(f"Warning: Ground-truth label directory not found: {GROUND_TRUTH_DIR}")
        print(f"Discovered test label files (.npy): {len(label_files)}")
        label_by_video_id = build_label_map(label_files)

        if len(test_sources) > 0 and len(label_files) > 0:
            matched = sum(1 for v in test_sources
                          if get_label_path_for_video(os.path.splitext(os.path.basename(v))[0]))
            print(f"Matched test videos to label files: {matched}/{len(test_sources)}")

    return train_sources, test_sources


label_by_video_id = {}
train_videos, test_sources = scan_dataset()

## Step 4: Loading, Preprocessing & Saving Helpers

Reusable core helpers, dataset-agnostic:

1. **Natural sort** — sort video/frame names numerically rather than lexicographically.
2. **Unified video decoder** — `decord` first with an OpenCV fallback for codecs `decord` cannot handle (carried over from the Avenue notebook).
3. **JPG-frame loader** — for the ShanghaiTech test split (folder of `.jpg` files).
4. **Clip index generator** — given a target frame `i` and total frames $N$, sample 16 frames with stride 4, centered around `i`:
   $$\text{clip\_indices}[k] = \text{clamp}(i - 30 + 4k, 0, N - 1) \quad \text{for } k=0,\dots,15$$
5. **Atomic NPZ saver** — write to a local SSD temp file, then copy to Drive (FUSE-safe).

In [ ]:
import re
import os
import glob
import tempfile
import shutil
import numpy as np
import torch
import cv2
from decord import VideoReader, cpu

# 1. Natural sorting helper (re-defined here for self-containment)
def natural_keys(text):
    return [int(c) if c.isdigit() else c.lower() for c in re.split(r'(\d+)', text)]

# 2. Unified video decoder: decord first, OpenCV fallback
def load_video(path):
    """Open a video file. Returns (reader, num_frames, fps).

    reader is a decord.VideoReader when decord can decode the file, otherwise a
    dict descriptor {"backend": "opencv", "path": path} for the OpenCV fallback.
    """
    try:
        vr = VideoReader(path, ctx=cpu(0))
        num_frames = len(vr)
        fps = float(vr.get_avg_fps())
        if not np.isfinite(fps) or fps <= 0:
            fps = 25.0
        return vr, num_frames, fps
    except Exception as decord_error:
        print(f"[WARNING] decord failed for {path}: {decord_error}")
        print("Falling back to OpenCV VideoCapture for this video.")
        cap = cv2.VideoCapture(path)
        if not cap.isOpened():
            raise ValueError(f"Failed to open video with decord and OpenCV: {path}") from decord_error
        num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        cap.release()
        if not np.isfinite(fps) or fps <= 0:
            fps = 25.0
        return {"backend": "opencv", "path": path}, num_frames, fps

# 3. ShanghaiTech test JPG-frame loader
def load_test_frames(folder_path):
    """Load a ShanghaiTech-style test folder (sequence of .jpg frames)."""
    frame_paths = sorted(glob.glob(os.path.join(folder_path, "*.jpg")), key=natural_keys)
    num_frames = len(frame_paths)
    fps = 24.0  # Standard fps for ShanghaiTech test sequences
    return frame_paths, num_frames, fps

# 4. Generate clip indices with temporal stride 4 centered around target frame i
def generate_clip_indices(i, num_frames):
    indices = []
    for k in range(16):
        idx = i - 30 + 4 * k
        idx = max(0, min(idx, num_frames - 1))
        indices.append(idx)
    return np.array(indices, dtype=np.int64)

# 5. Atomic save function for NPZ files (FUSE/Google Drive safe)
def save_npz_atomic(file_path, data_dict):
    dir_name = os.path.dirname(file_path)
    os.makedirs(dir_name, exist_ok=True)

    # Create temp file on the LOCAL Colab SSD filesystem (fast, robust POSIX writes)
    fd, temp_path = tempfile.mkstemp(suffix=".npz")
    try:
        os.close(fd)
        np.savez_compressed(temp_path, **data_dict)
        # Copy from local SSD to Google Drive (sequential FUSE-friendly write)
        shutil.copy2(temp_path, file_path)
        os.remove(temp_path)
    except Exception as e:
        if os.path.exists(temp_path):
            os.remove(temp_path)
        raise e

## Step 5: Load Hiera-L Model & Adapt for Feature Extraction

Download `hiera_large_16x224` (MAE pretrained, Kinetics-400 fine-tuned) from PyTorch Hub. To extract the raw pooled 1152-D spatiotemporal representations instead of classification logits, we:

1. Re-assign `model.head = torch.nn.Identity()`.
2. Switch the model to evaluation mode (`model.eval()`).
3. Move weights to the active CUDA device, if available.
4. Validate the output shape with a dummy tensor $[B, C, T, H, W] = [2, 3, 16, 224, 224]$, expecting $[B, 1152]$.

In [ ]:
def load_and_validate_hiera():
    print("Loading Hiera-L model (hiera_large_16x224, checkpoint=mae_k400_ft_k400) from PyTorch Hub...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using execution device: {device}")

    model = torch.hub.load(
        "facebookresearch/hiera",
        model="hiera_large_16x224",
        pretrained=True,
        checkpoint="mae_k400_ft_k400"
    )

    print(f"Original model head: {model.head}")

    # Replace the head with Identity to extract pooled features
    model.head = torch.nn.Identity()
    print("Modified model head to torch.nn.Identity() for feature extraction.")

    model = model.to(device)
    model.eval()

    # Validate with a dummy tensor of shape [B, C, T, H, W]
    dummy_input = torch.randn(2, 3, 16, 224, 224, device=device)
    with torch.no_grad():
        dummy_output = model(dummy_input)

    print(f"Dummy Input shape: {dummy_input.shape}")
    print(f"Dummy Output shape: {dummy_output.shape}")

    assert dummy_output.shape == (2, 1152), \
        f"Validation Error: Expected output shape [2, 1152], got {dummy_output.shape}"
    print("✓ Model output shape validation successful! Hiera-L correctly produces 1152-D spatiotemporal vectors.")

    return model, device

model, device = load_and_validate_hiera()

## Step 6: End-to-End Smoke Testing

Validate loading, preprocessing, model execution, indexing, and atomic NPZ saving on a single train item and a single test item. If raw data is not present, a synthetic mock extraction exercises the same code path under dry-run conditions.

In [ ]:
import os
import gc
import cv2
import json
import datetime
import tqdm
import shutil
import numpy as np
import torch

# =============================================================================
# Pre-caching functions: decode & preprocess every frame exactly ONCE.
# Eliminates the 16x redundant decoding that caused the original bottleneck.
# Memory: a 764-frame video cached at [764, 3, 224, 224] float32 ≈ 460 MB.
# =============================================================================

def preprocess_all_frames_from_video(video_reader, num_frames, target_size=(224, 224), chunk_size=128):
    """Pre-decode and preprocess ALL frames from a video (AVI/MP4).
    Uses chunked decord loading when available; falls back to sequential OpenCV
    reads when decord cannot decode the video.
    Returns numpy array [num_frames, 3, 224, 224] (float32, Hiera-normalized).
    """
    mean = np.array([0.45, 0.45, 0.45], dtype=np.float32).reshape(1, 3, 1, 1)
    std  = np.array([0.225, 0.225, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

    all_frames = np.empty((num_frames, 3, target_size[0], target_size[1]), dtype=np.float32)

    if hasattr(video_reader, "get_batch"):
        for start in range(0, num_frames, chunk_size):
            end = min(start + chunk_size, num_frames)
            indices = list(range(start, end))
            frames_np = video_reader.get_batch(indices).asnumpy()  # [chunk, H, W, C] RGB uint8
            for j, img in enumerate(frames_np):
                img_resized = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
                img_float = img_resized.astype(np.float32) / 255.0
                all_frames[start + j] = img_float.transpose(2, 0, 1)  # HWC -> CHW
            del frames_np
    elif isinstance(video_reader, dict) and video_reader.get("backend") == "opencv":
        cap = cv2.VideoCapture(video_reader["path"])
        try:
            for frame_idx in range(num_frames):
                ok, img_bgr = cap.read()
                if not ok:
                    raise ValueError(f"Failed to read frame {frame_idx} from {video_reader['path']}")
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                img_resized = cv2.resize(img_rgb, target_size, interpolation=cv2.INTER_LINEAR)
                img_float = img_resized.astype(np.float32) / 255.0
                all_frames[frame_idx] = img_float.transpose(2, 0, 1)  # HWC -> CHW
        finally:
            cap.release()
    else:
        raise TypeError("Unsupported video reader type.")

    all_frames = (all_frames - mean) / std
    return all_frames  # [num_frames, 3, 224, 224]


def preprocess_all_frames_from_jpgs(frame_paths, target_size=(224, 224)):
    """Pre-load and preprocess ALL JPG frames from a ShanghaiTech-style test folder.
    Returns numpy array [num_frames, 3, 224, 224] (float32, Hiera-normalized).
    """
    num_frames = len(frame_paths)
    mean = np.array([0.45, 0.45, 0.45], dtype=np.float32).reshape(1, 3, 1, 1)
    std  = np.array([0.225, 0.225, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

    all_frames = np.empty((num_frames, 3, target_size[0], target_size[1]), dtype=np.float32)
    for j, path in enumerate(frame_paths):
        img = cv2.imread(path)
        if img is None:
            raise ValueError(f"Failed to read image at {path}")
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img_rgb, target_size, interpolation=cv2.INTER_LINEAR)
        img_float = img_resized.astype(np.float32) / 255.0
        all_frames[j] = img_float.transpose(2, 0, 1)  # HWC -> CHW

    all_frames = (all_frames - mean) / std
    return all_frames


# =============================================================================
# Main extraction function (frame pre-caching, dynamic OOM recovery, FP16 autocast)
# =============================================================================

def extract_features_for_video(video_source, is_train, out_npz_path, model, device, batch_size=8):
    """
    Optimized feature extraction with pre-cached preprocessed frames.
    Each frame is decoded, resized, and normalized exactly ONCE; clips are then
    assembled via fast numpy index slicing.

    video_source:
      - a path to a video file (.avi/.mp4) for both splits on Avenue and the
        train split on ShanghaiTech, OR
      - a path to a folder of .jpg frames for the ShanghaiTech test split.
      The loader is chosen automatically via os.path.isdir(video_source).
    Includes dynamic CUDA OOM recovery (halve batch size and retry) and FP16
    autocast on CUDA. Returns (out_npz_path, num_frames, fps).
    """
    if video_source == 'mock':
        # Synthetic mock extraction (dry-run path)
        num_frames = 45
        fps = 24.0
        video_id = "mock_video_01"
        split = "train" if is_train else "test"
        source_path = "mock_source"

        print(f"Running mock feature extraction for {video_id} ({split})...")
        features = np.random.randn(num_frames, 1152).astype(np.float32)
        frame_indices = np.arange(num_frames, dtype=np.int64)
        clip_indices = np.zeros((num_frames, 16), dtype=np.int64)
        for i in range(num_frames):
            clip_indices[i] = generate_clip_indices(i, num_frames)
    else:
        split = "train" if is_train else "test"
        is_frame_folder = os.path.isdir(video_source)
        video_id = os.path.splitext(os.path.basename(video_source.rstrip("/")))[0]

        # Open the source
        if is_frame_folder:
            frame_paths, num_frames, fps = load_test_frames(video_source)
        else:
            reader, num_frames, fps = load_video(video_source)

        source_path = video_source
        print(f"Extracting features for {video_id} ({split})... Total frames: {num_frames}")

        # ---- PHASE 1: Pre-cache all preprocessed frames (each frame decoded ONCE) ----
        print(f"  Phase 1/2: Pre-caching {num_frames} preprocessed frames...")
        if is_frame_folder:
            cached_frames = preprocess_all_frames_from_jpgs(frame_paths)
        else:
            cached_frames = preprocess_all_frames_from_video(reader, num_frames)
            del reader
        gc.collect()
        mem_mb = cached_frames.nbytes / (1024 * 1024)
        print(f"  Frame cache ready: {cached_frames.shape} — {mem_mb:.1f} MB")

        # ---- PHASE 2: Assemble clips via slicing + batched GPU inference (OOM recovery) ----
        print(f"  Phase 2/2: Running batched Hiera-L inference...")
        frame_indices = np.arange(num_frames, dtype=np.int64)
        clip_indices = np.zeros((num_frames, 16), dtype=np.int64)
        for i in range(num_frames):
            clip_indices[i] = generate_clip_indices(i, num_frames)

        current_batch_size = batch_size
        success = False
        features_list = []

        while not success and current_batch_size >= 1:
            try:
                features_list = []
                for batch_start in tqdm.tqdm(range(0, num_frames, current_batch_size),
                                             desc=f"Batches (size={current_batch_size})"):
                    batch_end = min(batch_start + current_batch_size, num_frames)
                    batch_clips = []
                    for i in range(batch_start, batch_end):
                        clip_frames = cached_frames[clip_indices[i]]   # [16, 3, 224, 224]
                        clip_tensor = torch.from_numpy(clip_frames.copy()).permute(1, 0, 2, 3)
                        batch_clips.append(clip_tensor)

                    stacked = torch.stack(batch_clips, dim=0).to(device)
                    with torch.no_grad():
                        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                            feats = model(stacked)  # [B, 1152]
                        features_list.append(feats.float().cpu().numpy())
                    del stacked, batch_clips

                features = np.concatenate(features_list, axis=0)  # [num_frames, 1152]
                success = True
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"\n[WARNING] CUDA Out of Memory with batch_size={current_batch_size}. "
                          f"Retrying with batch_size={current_batch_size // 2}...")
                    current_batch_size //= 2
                    if 'stacked' in locals(): del stacked
                    if 'batch_clips' in locals(): del batch_clips
                    if 'features_list' in locals(): del features_list
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    if current_batch_size < 1:
                        raise e
                else:
                    raise e

        del cached_frames
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Package metadata
    metadata = {
        "dataset": DATASET_CHOICE,
        "model": MODEL_NAME,
        "checkpoint": CHECKPOINT_NAME,
        "hiera_commit": HIERA_COMMIT,
        "extraction_date": datetime.datetime.now().isoformat(),
        "pytorch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "temporal_stride": TEMPORAL_STRIDE,
        "clip_len": NUM_FRAMES_PER_CLIP,
        "preprocessing": {
            "resize": f"{IMAGE_SIZE}x{IMAGE_SIZE}",
            "mean": [0.45, 0.45, 0.45],
            "std": [0.225, 0.225, 0.225]
        }
    }

    data_to_save = {
        "features": features.astype(np.float32),
        "frame_indices": frame_indices.astype(np.int64),
        "clip_indices": clip_indices.astype(np.int64),
        "video_id": video_id,
        "split": split,
        "source_path": source_path,
        "num_frames": int(num_frames),
        "fps": float(fps),
        "metadata_json": json.dumps(metadata, indent=2)
    }

    save_npz_atomic(out_npz_path, data_to_save)
    print(f"✓ Saved: {out_npz_path}")
    return out_npz_path, int(num_frames), float(fps)


# Execute Smoke Test
print("--- Launching Smoke Test ---")
local_smoke_train_out = "/content/smoke_test_train_local.npz"
local_smoke_test_out  = "/content/smoke_test_test_local.npz"
smoke_train_out = os.path.join(TRAIN_FEATURE_DIR, "smoke_test_video.npz")
smoke_test_out  = os.path.join(TEST_FEATURE_DIR, "smoke_test_video.npz")

if len(train_videos) > 0 and len(test_sources) > 0:
    print(f"Real {DATASET_CHOICE} dataset files discovered. Running real smoke test...")
    extract_features_for_video(train_videos[0], is_train=True,  out_npz_path=local_smoke_train_out,
                               model=model, device=device)
    extract_features_for_video(test_sources[0], is_train=False, out_npz_path=local_smoke_test_out,
                               model=model, device=device)
else:
    print("No raw dataset files detected at paths. Executing mock smoke test...")
    extract_features_for_video("mock", is_train=True,  out_npz_path=local_smoke_train_out,
                               model=model, device=device)
    extract_features_for_video("mock", is_train=False, out_npz_path=local_smoke_test_out,
                               model=model, device=device)

# Verify reloading and statistics on LOCAL SSD paths
for path in [local_smoke_train_out, local_smoke_test_out]:
    assert os.path.exists(path), f"Smoke Test Failure: Output file {path} was not created."
    loaded = np.load(path)
    print(f"\nVerifying loaded file: {os.path.basename(path)}")
    print(f" - features shape: {loaded['features'].shape}")
    print(f" - num_frames: {loaded['num_frames']}")
    print(f" - video_id: {loaded['video_id']}")
    assert np.isfinite(loaded['features']).all(), \
        "Smoke Test Failure: Features contain non-finite values (NaN/Inf)."
    assert loaded['features'].shape[1] == 1152, \
        f"Smoke Test Failure: Expected feature size 1152, got {loaded['features'].shape[1]}"
    print(" - NaN/Inf check: Passed (all values finite)")
    print(" - Feature dimensions check: Passed")
    meta = json.loads(str(loaded['metadata_json']))
    print(f" - Metadata loaded: dataset={meta['dataset']}, model={meta['model']}, date={meta['extraction_date']}")

print("\nCopying verified smoke test artifacts to Google Drive...")
shutil.copy2(local_smoke_train_out, smoke_train_out)
shutil.copy2(local_smoke_test_out, smoke_test_out)
print(f"✓ Saved to Drive: {smoke_train_out}")
print(f"✓ Saved to Drive: {smoke_test_out}")
print("\n✓ Smoke test executed and validated successfully!")

## Step 7: Full Dataset Feature Extraction (with Resume Support)

Run the main extraction loops over all training and testing sources. Robustness features:

1. **Resume support** — skip any source whose target `.npz` already exists and validates.
2. **Atomic writes** — write to a temp file first, then copy to Drive (no partial/corrupt files).
3. **Manifest compilation** — track per-source status, frame counts, fps, and (Avenue only) matched label paths.

In [ ]:
import os
import traceback
import numpy as np
import pandas as pd


def verify_existing_npz(path):
    """Return True if path exists and is a valid non-corrupt npz with required keys."""
    if not os.path.exists(path):
        return False
    try:
        loaded = np.load(path)
        required_keys = ['features', 'frame_indices', 'clip_indices', 'video_id', 'split', 'num_frames']
        for k in required_keys:
            if k not in loaded:
                return False
        if loaded['features'].ndim != 2 or loaded['features'].shape[1] != 1152:
            return False
        return True
    except Exception:
        return False


def run_full_extraction(train_sources, test_sources, model, device, batch_size=16):
    print("--- Initiating Full Feature Extraction Pipeline ---")

    train_manifest_rows = []
    test_manifest_rows = []

    # 1. Process Training Videos (always .avi)
    if len(train_sources) == 0:
        print("No raw training videos found to extract. Skipping train split extraction loop.")
    else:
        print(f"Starting Train split extraction: {len(train_sources)} videos...")
        for i, video_path in enumerate(train_sources):
            video_id = os.path.splitext(os.path.basename(video_path))[0]
            feature_path = os.path.join(TRAIN_FEATURE_DIR, f"{video_id}.npz")
            row = {
                "split": "train", "video_id": video_id, "source_path": video_path,
                "feature_path": feature_path, "label_path": "",
                "num_frames": 0, "fps": 0.0, "status": "pending",
            }
            try:
                if verify_existing_npz(feature_path):
                    print(f"[{i+1}/{len(train_sources)}] Skipping (valid NPZ already exists): {video_id}")
                    loaded = np.load(feature_path)
                    row["num_frames"] = int(loaded["num_frames"])
                    row["fps"] = float(loaded["fps"])
                    row["status"] = "success"
                else:
                    print(f"[{i+1}/{len(train_sources)}] Extracting features for: {video_id}")
                    _, num_frames, fps = extract_features_for_video(
                        video_path, is_train=True, out_npz_path=feature_path,
                        model=model, device=device, batch_size=batch_size)
                    row["num_frames"] = num_frames
                    row["fps"] = fps
                    row["status"] = "success"
            except Exception as e:
                print(f"Error extracting features for video {video_id}: {str(e)}")
                traceback.print_exc()
                row["status"] = f"error: {str(e)}"
            train_manifest_rows.append(row)

    # 2. Process Testing Sources (Avenue: .avi; ShanghaiTech: JPG folders)
    if len(test_sources) == 0:
        print("No raw testing sources found to extract. Skipping test split extraction loop.")
    else:
        print(f"\nStarting Test split extraction: {len(test_sources)} sources...")
        for i, source_path in enumerate(test_sources):
            # video_id: stem of file (Avenue) or folder name (ShanghaiTech)
            video_id = os.path.splitext(os.path.basename(source_path.rstrip("/")))[0]
            feature_path = os.path.join(TEST_FEATURE_DIR, f"{video_id}.npz")
            label_path = ""
            if DATASET_CHOICE == "Avenue":
                label_path = get_label_path_for_video(video_id)

            row = {
                "split": "test", "video_id": video_id, "source_path": source_path,
                "feature_path": feature_path, "label_path": label_path,
                "num_frames": 0, "fps": 0.0, "status": "pending",
            }
            try:
                if verify_existing_npz(feature_path):
                    print(f"[{i+1}/{len(test_sources)}] Skipping (valid NPZ already exists): {video_id}")
                    loaded = np.load(feature_path)
                    row["num_frames"] = int(loaded["num_frames"])
                    row["fps"] = float(loaded["fps"])
                    row["status"] = "success"
                else:
                    print(f"[{i+1}/{len(test_sources)}] Extracting features for: {video_id}")
                    _, num_frames, fps = extract_features_for_video(
                        source_path, is_train=False, out_npz_path=feature_path,
                        model=model, device=device, batch_size=batch_size)
                    row["num_frames"] = num_frames
                    row["fps"] = fps
                    row["status"] = "success"
            except Exception as e:
                print(f"Error extracting features for test source {video_id}: {str(e)}")
                traceback.print_exc()
                row["status"] = f"error: {str(e)}"
            test_manifest_rows.append(row)

    # 3. Save manifests
    if len(train_manifest_rows) > 0:
        pd.DataFrame(train_manifest_rows).to_csv(MANIFEST_TRAIN, index=False)
        print(f"\n✓ Saved training manifest to: {MANIFEST_TRAIN}")
    if len(test_manifest_rows) > 0:
        pd.DataFrame(test_manifest_rows).to_csv(MANIFEST_TEST, index=False)
        print(f"✓ Saved testing manifest to: {MANIFEST_TEST}")

    print("\n--- Feature Extraction Cycle Complete ---")
    return train_manifest_rows, test_manifest_rows


train_rows, test_rows = run_full_extraction(train_videos, test_sources, model, device)

## Step 8: Compute Training Set Feature Statistics (Mean & Std)

MULDE trains on standardized features. Because loading the entire training set (up to ~300k frames) at once would exhaust RAM, we compute the global mean and standard deviation with a **memory-efficient streaming algorithm**: read each train `.npz` sequentially, accumulate sums and squared sums, then derive:

- **Global Mean Vector**: `[1152]`
- **Global Standard Deviation Vector**: `[1152]`

Saved as `train_feature_stats.npz` for downstream standardization.

In [ ]:
import os
import glob
import tqdm
import datetime
import numpy as np


def compute_train_statistics():
    print("--- Computing Global Training Set Feature Statistics ---")

    all_train_files = sorted(glob.glob(os.path.join(TRAIN_FEATURE_DIR, "*.npz")))
    all_train_files = [f for f in all_train_files if "smoke_test" not in os.path.basename(f)]

    if len(all_train_files) == 0:
        print("No real training feature files found! Checking for smoke test file...")
        all_train_files = sorted(glob.glob(os.path.join(TRAIN_FEATURE_DIR, "smoke_test_video.npz")))
        if len(all_train_files) == 0:
            print("No training feature files found! Cannot calculate statistics.")
            return None

    print(f"Streaming through {len(all_train_files)} training feature files...")

    n_frames = 0
    sum_x  = np.zeros(1152, dtype=np.float64)
    sum_x2 = np.zeros(1152, dtype=np.float64)

    for path in tqdm.tqdm(all_train_files, desc="Streaming Stats"):
        try:
            data = np.load(path)
            feats = data['features'].astype(np.float64)  # [num_frames, 1152]
            sum_x  += feats.sum(axis=0)
            sum_x2 += (feats ** 2).sum(axis=0)
            n_frames += feats.shape[0]
        except Exception as e:
            print(f"Skipping file {path} due to load error: {e}")

    if n_frames == 0:
        print("Total loaded training frames is 0. Cannot compute stats.")
        return None

    global_mean = sum_x / n_frames
    global_variance = (sum_x2 / n_frames) - (global_mean ** 2)
    global_variance = np.clip(global_variance, 0.0, None)  # clamp tiny negatives
    global_std = np.sqrt(global_variance)

    zero_var_mask = global_std < 1e-8
    if zero_var_mask.any():
        print(f"Warning: Found {zero_var_mask.sum()} feature dimensions with near-zero variance. "
              f"Fixing std to 1.0 for these features.")
        global_std[zero_var_mask] = 1.0

    print(f"\nSuccessfully accumulated statistics over {n_frames} frames.")
    print(f"Mean Vector - min: {global_mean.min():.5f}, max: {global_mean.max():.5f}, avg: {global_mean.mean():.5f}")
    print(f"Std Vector  - min: {global_std.min():.5f}, max: {global_std.max():.5f}, avg: {global_std.mean():.5f}")

    stats_dict = {
        "mean": global_mean.astype(np.float32),
        "std": global_std.astype(np.float32),
        "num_frames": int(n_frames),
        "computed_at": datetime.datetime.now().isoformat()
    }
    save_npz_atomic(FEATURE_STATS_PATH, stats_dict)
    print(f"✓ Saved training feature statistics to: {FEATURE_STATS_PATH}")
    return FEATURE_STATS_PATH


stats_path = compute_train_statistics()

## Step 9: Generate Consolidated MULDE-Ready Config Index

Compile a unified `extraction_config.json` capturing model specs, preprocessing details, spatiotemporal sampling, artifact locations, and execution environment. The dataset field and raw-inputs section reflect the selected dataset.

In [ ]:
import os
import json
import datetime
import torch


def generate_mulde_config_index():
    print("--- Creating Consolidated MULDE Config Index ---")

    raw_inputs = {
        "archive_path": ARCHIVE_PATH,
        "archive_type": ARCHIVE_TYPE,
        "extract_root": EXTRACT_ROOT,
        "train_video_directory": TRAIN_VIDEO_DIR,
    }
    if DATASET_CHOICE == "Avenue":
        raw_inputs.update({
            "test_video_directory": TEST_VIDEO_DIR,
            "ground_truth_directory": GROUND_TRUTH_DIR,
        })
    else:
        raw_inputs.update({"test_frame_directory": TEST_FRAME_DIR})

    config_dict = {
        "dataset": DATASET_CHOICE,
        "raw_inputs": raw_inputs,
        "model_name": MODEL_NAME,
        "checkpoint_name": CHECKPOINT_NAME,
        "hiera_github_commit": HIERA_COMMIT,
        "feature_dimension": 1152,
        "spatiotemporal_sampling": {
            "clip_length": NUM_FRAMES_PER_CLIP,
            "temporal_stride": TEMPORAL_STRIDE,
            "alignment": "centered_around_target",
            "frame_selection_formula": "clamp(target - 30 + 4*k, 0, N-1) for k=0..15"
        },
        "preprocessing_pipeline": {
            "resize_dimensions": [IMAGE_SIZE, IMAGE_SIZE],
            "crop_policy": "full_frame_resize",
            "color_space": "RGB",
            "intensity_scaling": "[0.0, 1.0]",
            "hiera_normalization_stats": {
                "mean": [0.45, 0.45, 0.45],
                "std": [0.225, 0.225, 0.225]
            }
        },
        "artifacts_registry": {
            "root_feature_directory": FEATURE_DIR,
            "relative_train_directory": "train",
            "relative_test_directory": "test",
            "train_manifest_csv": os.path.basename(MANIFEST_TRAIN),
            "test_manifest_csv": os.path.basename(MANIFEST_TEST),
            "train_feature_stats_npz": os.path.basename(FEATURE_STATS_PATH)
        },
        "system_metadata": {
            "pytorch_version": torch.__version__,
            "torch_cuda_available": torch.cuda.is_available(),
            "creation_time": datetime.datetime.now().isoformat()
        }
    }

    try:
        with open(CONFIG_PATH, "w") as f:
            json.dump(config_dict, f, indent=2)
        print(f"✓ Consolidated configuration successfully written to: {CONFIG_PATH}")
        print("\n--- Consolidated Extraction Configuration JSON ---")
        print(json.dumps(config_dict, indent=2))
    except Exception as e:
        print(f"Error saving consolidated configuration: {e}")


generate_mulde_config_index()

## Pipeline Execution & Verification Plan

### Automated Verification
Once this notebook is executed in a Colab session:

1. **Smoke Test Output Verification**
   - `train/smoke_test_video.npz` and `test/smoke_test_video.npz` exist with `features` of shape `[num_frames, 1152]`.
   - Features contain no `NaN`/`Inf` and have non-zero variance.
2. **Boundary Clamping Verification**
   - For frame `0`, the clip indices start with eight `0`s then `2, 6, 10, 14, 18, 22, 26, 30` (clamped to `N-1` when `N` is small).
   - For frame `i = num_frames - 1`, indices clamp correctly to `num_frames - 1`.
3. **Manifest & Index Integrity**
   - `manifest_train.csv` / `manifest_test.csv` exist under the dataset-specific feature directory.
   - Every successful row points to an existing `.npz` file.
   - For Avenue, the test manifest includes matched `label_path` values when `ground_truth_avenue` is available.
4. **Statistics & Configuration**
   - `train_feature_stats.npz` contains valid `mean`/`std` vectors of dimension `1152`.
   - `extraction_config.json` points to the correct dataset-specific feature directory, manifests, stats, and raw inputs.

### Expected Final Output Layout

```text
/content/drive/MyDrive/MULDE/<feature_subdir>/   # dataset-specific
├── train/
│   ├── <train_video_id>.npz
│   └── ...
├── test/
│   ├── <test_video_id>.npz
│   └── ...
├── manifest_train.csv
├── manifest_test.csv
├── train_feature_stats.npz
└── extraction_config.json
```

### Loading Extracted Features in MULDE Training
```python
import numpy as np
import torch

def load_standardized_features(npz_path, stats_path):
    data  = np.load(npz_path)
    stats = np.load(stats_path)
    raw_features = data['features']      # [num_frames, 1152]
    mean = stats['mean']                 # [1152]
    std  = stats['std']                  # [1152]
    standardized = (raw_features - mean) / (std + 1e-8)
    return torch.tensor(standardized, dtype=torch.float32)
```